In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder,LabelEncoder,StandardScaler,MinMaxScaler

In [2]:
df = pd.read_csv('titanic_data_updated.csv')
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,no,third,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,yes,first,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,yes,third,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,yes,first,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,no,third,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,no,second,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,yes,first,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,no,third,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,yes,first,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


### splitting data

In [3]:
df.drop(['PassengerId','Name','Ticket'],axis=1,inplace=True)

# family_size creation
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

#feature and target extract
X = df.drop(['Survived'],axis=1)
y = df['Survived']


X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

### imputation of data

In [4]:
imputer_transformer = ColumnTransformer(
    transformers=[
        ('age',SimpleImputer(missing_values=np.nan , strategy='mean'),['Age']),
        ('embarked',SimpleImputer(missing_values=np.nan , strategy='most_frequent'),['Embarked']),
        ('cabin',SimpleImputer(missing_values=np.nan , strategy='constant',fill_value='Missing',add_indicator=True),['Cabin'])
    ],
    remainder='passthrough',
    verbose_feature_names_out = False
)
imputer_transformer.set_output(transform='pandas')

imputer_transformer.fit(X_train)

X_train = imputer_transformer.transform(X_train)
X_test = imputer_transformer.transform(X_test)

### outlier handling of age

In [5]:
mean_of_age = X_train['Age'].mean()
std_of_age = X_train['Age'].std()
X_train['zscore_Age'] = (X_train['Age']-mean_of_age)/std_of_age

X_train = X_train[abs(X_train['zscore_Age']) <=3]
X_train.drop(['zscore_Age'],axis=1 , inplace=True)

### Outlier handling of Fare

In [6]:
fare_Q1 = X_train['Fare'].quantile(0.25)
fare_Q3 = X_train['Fare'].quantile(0.75)
fare_IQR = fare_Q3 - fare_Q1
fare_minimum = max(0,fare_Q1 - 1.5 * fare_IQR)
fare_maximum = fare_Q3 + 1.5 * fare_IQR

X_train['Fare']= X_train['Fare'].clip(fare_minimum , fare_maximum)

### encoding and scaling

In [7]:
X_train['Cabin_Deck'] = X_train['Cabin'].astype(str).str[0]
X_test['Cabin_Deck'] = X_test['Cabin'].astype(str).str[0]

In [8]:
encoder_scaler = ColumnTransformer(
    transformers=[
        ('pclass',OrdinalEncoder(categories=[['third','second','first']]),['Pclass']),
        ('embarked_sex',OneHotEncoder(sparse_output=False,drop='first'),['Embarked','Sex','Cabin_Deck']),
        ('age_scaler',StandardScaler(),['Age']),
        ('fare_scaler',MinMaxScaler(),['Fare','FamilySize'])
    ],
    remainder='passthrough',
    verbose_feature_names_out = False
)
encoder_scaler.set_output(transform='pandas')

encoder_scaler.fit(X_train)

X_train = encoder_scaler.transform(X_train)
X_test = encoder_scaler.transform(X_test)

In [9]:
X_train.drop(['Cabin','SibSp','Parch'],axis=1,inplace=True)
X_test.drop(['Cabin','SibSp','Parch'],axis=1,inplace=True)

### Final Dataset

In [10]:
X_train

,Pclass,Embarked_Q,Embarked_S,Sex_male,Cabin_Deck_B,Cabin_Deck_C,Cabin_Deck_D,Cabin_Deck_E,Cabin_Deck_F,Cabin_Deck_G,Cabin_Deck_M,Cabin_Deck_T,Age,Fare,FamilySize,missingindicator_Cabin
331,2.0,0.0,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.304495,0.442804,0.0,False
733,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.495295,0.201981,0.0,True
382,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.224621,0.123131,0.0,True
704,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.255323,0.122031,0.1,True
813,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.855135,0.485920,0.6,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.655276,0.118858,0.0,True
270,2.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.024552,0.481647,0.0,True
860,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.944537,0.219201,0.2,True
435,2.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.215210,1.000000,0.3,False


In [11]:
X_test

,Pclass,Embarked_Q,Embarked_S,Sex_male,Cabin_Deck_B,Cabin_Deck_C,Cabin_Deck_D,Cabin_Deck_E,Cabin_Deck_F,Cabin_Deck_G,Cabin_Deck_M,Cabin_Deck_T,Age,Fare,FamilySize,missingindicator_Cabin
709,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.024552,0.236874,0.2,True
439,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.144630,0.163138,0.0,True
840,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.735267,0.123131,0.0,True
720,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.855135,0.512721,0.1,True
39,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.215210,0.174662,0.1,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
433,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.975238,0.110701,0.0,True
773,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.024552,0.112255,0.0,True
25,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.704565,0.487668,0.6,True
84,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.975238,0.163138,0.0,True
